In [4]:
!pip install ultralytics kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.0/47.0 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.6/97.6 kB 11.4 MB/s eta 0:00:00


In [5]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [7]:
import os
import shutil
import glob
import pandas as pd
from sklearn.model_selection import train_test_split
from ultralytics import YOLO
import kagglehub

os.environ['KAGGLE_API_TOKEN'] = "KGAT_0c5e062c4d622e8403def82b24080c49"


path = kagglehub.dataset_download("kmader/skin-cancer-mnist-ham10000")

dataset_dir = "/content/yolo_dataset"
os.makedirs(dataset_dir, exist_ok=True)

df = pd.read_csv(f"{path}/HAM10000_metadata.csv")
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['dx'])

all_images = glob.glob(f"{path}/**/*.jpg", recursive=True)
img_dict = {os.path.splitext(os.path.basename(p))[0]: p for p in all_images}

def organize_images(df_split, split_name):
    for _, row in df_split.iterrows():
        img_id = row['image_id']
        label = row['dx']
        src = img_dict.get(img_id)
        if src:
            dst_dir = f"{dataset_dir}/{split_name}/{label}"
            os.makedirs(dst_dir, exist_ok=True)
            shutil.copy(src, f"{dst_dir}/{img_id}.jpg")

organize_images(train_df, "train")
organize_images(val_df, "val")

model = YOLO("yolov8n-cls.pt")
results = model.train(data=dataset_dir, epochs=20, imgsz=224)

model.export(format='onnx')

try:
    shutil.copy('/content/runs/classify/train/weights/best.pt', '/content/drive/MyDrive/')
    shutil.copy('/content/runs/classify/train/weights/best.onnx', '/content/drive/MyDrive/')
    print("成功")
except Exception as e:
    print("發生錯誤", e)

Using Colab cache for faster access to the 'skin-cancer-mnist-ham10000' dataset.
Ultralytics 8.4.183 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/yolo_dataset, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi